In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


from datetime import datetime
import pytz
def percent_nan_per_col(df, column):
        return (df[column].isnull().sum() / len(df)) * 100


def compare_datetimes(publishedDate, eventDate):
    # Parse first date: "12/2/2022 0:00" (assumed to be local or naive)
    date1 = datetime.strptime(eventDate.strip(), "%m/%d/%Y %H:%M")
    date1 = date1.replace(tzinfo=pytz.UTC)  # Assume UTC; adjust if needed

    # Parse second date: "2020-12-31T18:15:00Z" (ISO 8601 UTC format)
    date2 = datetime.strptime(publishedDate, "%Y-%m-%dT%H:%M:%SZ")
    date2 = date2.replace(tzinfo=pytz.UTC)

    # Calculate time difference
    delta = date1 - date2

    # Return time difference in a friendly format
    return delta.days

#{
#        "days": delta.days,
#        "seconds": delta.total_seconds(),
#        "hours": delta.total_seconds() / 3600,
#    }

In [ ]:
folderPC = "C:/Users/Michael/Desktop/CS-513/Final Project/"
folderLaptop = "C:/Users/Michael/OneDrive - stevens.edu/Desktop/CS-513/Final Project/"
folder = folderPC

In [ ]:
df = pd.read_csv(folder + "flight_cases.csv", na_values="", keep_default_na=False)

total_rows = len(df)

df = df.drop(columns=["Source"])
df = df.dropna(how='all')
df = df.reset_index()

print(df.describe())
df = df.drop(columns=["DocketUrl", "ReportUrl", "RepGenFlag", "ProbableCause", "Findings", "N#", "Mkey", "SerialNumber", "OriginalPublishedDate", "DocketOriginalPublishedDate" ])
print(df.dtypes)
print(df.describe())
df.tail(5)

### Dropping cols with > 40% null

In [ ]:
cols_to_drop = []
for i in df.columns:
    n = percent_nan_per_col(df, i)
    if (n > 40):
        print(i + ": " + f"{n:.2f}" + "\n")
        cols_to_drop.append(i)
df = df.drop(columns=cols_to_drop)
print(df.columns)
        



### Cleaning bad records

In [ ]:
# NtsbNo

for index, row in df.iterrows():
    if ((len(row.NtsbNo) != 10) or ("." in row.NtsbNo) or (row.NtsbNo == None)):
        print(index)
        df = df.drop(index)
df = df.reset_index()

total_rows_dropped = total_rows - len(df)

# Maybe do more cleaning, but not enough time, need to focus on model prep for now.
    

In [ ]:
df.to_csv("clean_flight_cases.csv", index=False)

In [ ]:
dtypes = {"NtsbNo": "str",
    "EventType": "str",
    "EventDate": "str",
    "City": "str",
    "State": "str",
    "Country": "str",
    "HasSafetyRec": "bool",
    "Mode": "str",
    "ReportType": "str",
    "HighestInjuryLevel": "object",
    "FatalInjuryCount": "float64",
    "SeriousInjuryCount": "float64",
    "MinorInjuryCount": "float64",
    "OnboardInjuryCount": "float64",
    "Latitude": "float64",
    "Longitude": "float64",
    "Make": "str",
    "Model": "str",
    "AirCraftCategory": "str",
    "AirportID": "str",
    "AmateurBuilt": "str",
    "NumberOfEngines": "str",
    "EngineType": "str",
    "PurposeOfFlight": "str",
    "FAR": "str",
    "AirCraftDamage": "str",
    "WeatherCondition": "str",
    "BroadPhaseofFlight": "str",
    "ReportStatus": "str",
    "MostRecentReportType": "str" }

df2 = pd.read_csv(folder + "clean_flight_cases.csv", na_values=["", " "], keep_default_na=True, dtype=dtypes)

df2 = df2.drop(columns=['index', 'ReportType', "NtsbNo", "DocketOriginalPublishedDate", "City", "State", "Country", "Latitude", "Longitude"])

print(df2.describe())
print(df2.head())

## Data Transformation

In [ ]:
str_cols = ["EventType","EventDate","Mode","OriginalPublishedDate","HighestInjuryLevel","Make","Model","AirCraftCategory","AirportID","EngineType","PurposeOfFlight","FAR","AirCraftDamage","WeatherCondition","BroadPhaseofFlight","ReportStatus","MostRecentReportType"]
int_cols = ["FatalInjuryCount","SeriousInjuryCount","MinorInjuryCount","OnboardInjuryCount", "NumberOfEngines"]

df3 = pd.read_csv(folder + "commas_removed_flight_cases.csv")
for col in str_cols:
    df3[col] = df3[col].astype('string')

df3['NumberOfEngines'] = df3['NumberOfEngines'].astype(float)
df3['AmateurBuilt'] = df3['AmateurBuilt'].astype(bool)
df3.dtypes

### Cleaning Commas

In [ ]:
# Transforms inputs like "1, 2" to just "1"
# Selecting which one 
def clean_commas(col, name):
    cleanedCol = []
    for value in col:
        if pd.isna(value):
            cleanedCol.append(pd.NA)
            continue
        if type(value) == str:
            # Split by comma, strip spaces, convert to set to remove duplicates
            parts = [part.strip() for part in str(value).split(",")]
            unique_parts = list(set(parts))

            if len(unique_parts) == 1:
                cleanedCol.append(unique_parts[0])
            else:
                # If values are inconsistent (e.g. "1, 2"), decide how to handle
                # For now, return first value
                cleanedCol.append(unique_parts[0])
        elif type(value) == int or float:
            cleanedCol.append(value)
    return cleanedCol

dfCommas = pd.DataFrame()

for col in df3.columns:
    if col == 'HasSafetyRec' or 'AmateurBuilt':
        continue
    df3[col] = clean_commas(df3[col], col)

print(dfCommas.head(20))
print(dfCommas.describe())


In [ ]:
# Checking for commas
cols = []
commas = {}
for index, row in df3.iterrows():
    for col in df3.columns:
        x = row[col]
        if type(x) == str:
            if ',' in x:
                if col not in cols: 
                    cols.append(col)
                commas[col] = x
        else:
            print(x)
print(commas)
print(cols)
print(len(df3))

In [ ]:
df3.to_csv("commas_removed_flight_cases.csv")

### Need to encode strings and normalize numerical data
### Ensure HighestInjuryLevel is there based on InjuryCounts

In [ ]:
df3.to_csv("normalized_data.csv", index=False)

In [ ]:
# Find general proportions of each value for each column
from collections import Counter

df3 = pd.read_csv(folder + "clean_unlabeled_data.csv")

counts = []
for column in df3.columns:
    if column == 'index':
        continue
    counts.append((column, dict(Counter(df3[column]))))

resArr = []
for tup in counts:
    dct = tup[1]
    tot = 0
    res = {}
    for key in dct:
        tot += dct[key]
    for key in dct:
        res[key] = float("{:.2f}".format(dct[key] / tot * 100))
    resArr.append((tup[0], res))


for i in resArr:
    try:
        plt.bar(i[1].keys(), i[1].values(), width=0.1)
        plt.title(i[0])
        plt.show()
    except TypeError:
        continue
        
    
    
    

In [ ]:
from collections import Counter

count = Counter(df['BroadPhaseofFlight'])
for key in sorted(count.keys()):
    print(key, ": ", count[key])

In [ ]:
from collections import Counter
types = []
vals = []
for i in df3['NumberOfEngines']:
    if type(i) == str:
        types.append(type(i))
        vals.append(i)
print(types)
print(vals)
countr = Counter(vals)
print(countr)

In [ ]:
from collections import Counter
for col in df2.columns:
    obj = []
    for i in df2[col]:
        obj.append(type(i).__name__)
    counts = Counter(obj)
    if (len(counts) > 0):
        print(col + ": " + str(counts))





In [ ]:
df_num = df2.select_dtypes(exclude='object')
df_obj = df2.select_dtypes(include='object')

print(df2.dtypes)

f = pd.get_dummies(df2['HasSafetyRec'])
print(f.value_counts())

In [ ]:
attr = df2.drop(columns=["HighestInjuryLevel"])
target = df2["HighestInjuryLevel"]

attr_train, attr_test, target_train, target_test = train_test_split(attr, target, test_size=0.2, random_state=77)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100,criterion='entropy', random_state=77)

model.fit(attr_train, target_train)
target_pred = model.predict(attr_test)
